In [ ]:
import os
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.models import load_model
from patchify import patchify, unpatchify
from skimage.morphology import skeletonize
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
def f1(y_true, y_pred):
    def recall_m(y_true, y_pred):
        TP = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
        Positives = K.sum(K.round(K.clip(y_true, 0, 1)))
        return TP / (Positives + K.epsilon())

    def precision_m(y_true, y_pred):
        TP = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
        Pred_Positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
        return TP / (Pred_Positives + K.epsilon())

    precision = precision_m(y_true, y_pred)
    recall = recall_m(y_true, y_pred)
    return 2 * ((precision * recall) / (precision + recall + K.epsilon()))

def detect_and_crop_petri_dish(image_path):
    image = cv2.imread(image_path, 0)
    original_image = cv2.imread(image_path)
    blurred = cv2.medianBlur(image, 5)
    _, binary_image = cv2.threshold(blurred, 150, 255, cv2.THRESH_BINARY)
    kernel = np.ones((9, 9), np.uint8)
    closed = cv2.erode(cv2.dilate(binary_image, kernel, iterations=3), kernel, iterations=4)
    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour)
    size = max(w, h)
    x_center, y_center = x + w // 2, y + h // 2
    x_start, y_start = x_center - size // 2, y_center - size // 2
    x_end, y_end = x_start + size, y_start + size
    cropped_image = original_image[max(0, y_start):min(image.shape[0], y_end),
                                   max(0, x_start):min(image.shape[1], x_end)]
    return cropped_image, (max(0, x_start), max(0, y_start), min(image.shape[1], x_end), min(image.shape[0], y_end))

def align_mask_to_original(predicted_mask, original_image, crop_coordinates):
    h, w = original_image.shape[:2]
    aligned_mask = np.zeros((h, w), dtype=np.float32)
    x_start, y_start, x_end, y_end = crop_coordinates
    adjustment = 100
    x_end += adjustment
    y_end += adjustment
    cropped_h, cropped_w = y_end - y_start, x_end - x_start
    resized_mask = cv2.resize(predicted_mask, (cropped_w, cropped_h), interpolation=cv2.INTER_LINEAR)
    aligned_mask[y_start:y_end, x_start:x_end] = resized_mask
    return aligned_mask

---

In [2]:
import os
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.models import load_model
from patchify import patchify, unpatchify
from skimage.morphology import skeletonize
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt

# Paths
dataset_dir = "Kaggle"  # Directory containing the input images
results_dir = "kaggle7_pi"  # Directory where results (CSV and masks) will be saved
os.makedirs(results_dir, exist_ok=True)  # Create the results directory if it doesn't exist

# Load the trained model
model_file = "anastasiia_234301_unet_model_256px.h5"  # Path to the trained U-Net model
model = load_model(model_file, custom_objects={"f1": lambda y_true, y_pred: y_true})  # Load the model

# Parameters
patch_size = 256  # Size of patches for model prediction
plants_per_image = 5  # Number of plants per image (fixed for this dataset)

# Utility Functions
def petri_dish_border(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    _, binary_img = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        raise RuntimeError("No contours found in the image.")
    largest = max(contours, key=cv2.contourArea)  # Find the largest contour
    x, y, w, h = cv2.boundingRect(largest)  # Get bounding box around the largest contour
    return image[y:y + h, x:x + w]

def mask_morph(binary_mask):
    kernel = np.ones((3, 3), np.uint8)  # Define a smaller kernel for thin roots
    mask = cv2.dilate(binary_mask, kernel, iterations=1)  # Dilate to connect segments
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)  # Close small gaps
    return (mask > 0).astype(np.uint8)

def mask_split(mask, width, num_sections):
    section_width = width // num_sections
    root_masks = []
    for i in range(num_sections):
        start_x = i * section_width
        end_x = (i + 1) * section_width if i < num_sections - 1 else width
        root_masks.append(mask[:, start_x:end_x])
    return root_masks

def root_skeleton_length(root_mask):
    skeleton = skeletonize(root_mask > 0)  # Skeletonize the binary mask
    coordinates = np.column_stack(np.nonzero(skeleton))  # Get skeleton points
    if len(coordinates) == 0:
        return 0  # Return 0 if no skeleton exists
    
    graph = nx.Graph()
    coord_map = {tuple(coord): idx for idx, coord in enumerate(coordinates)}
    
    for coord in coordinates:
        y, x = coord
        graph.add_node(coord_map[(y, x)], coord=(y, x))
        for dy, dx in [(-1, 0), (1, 0), (0, -1), (0, 1), (-1, -1), (-1, 1), (1, -1), (1, 1)]:
            neighbor = (y + dy, x + dx)
            if neighbor in coord_map:
                dist = np.sqrt(dy**2 + dx**2)  # Euclidean distance
                graph.add_edge(coord_map[(y, x)], coord_map[neighbor], weight=dist)

    lengths = []
    for component in nx.connected_components(graph):
        subgraph = graph.subgraph(component)
        nodes = list(subgraph.nodes)
        if nodes:
            start_node = nodes[0]
            distances, _ = nx.single_source_dijkstra(subgraph, start_node, weight="weight")
            lengths.append(max(distances.values()))

    return max(lengths, default=0)

def predict_roots(image, model, save_path, image_name):
    h, w, _ = image.shape
    pad_h = (patch_size - h % patch_size) % patch_size  # Calculate padding for height
    pad_w = (patch_size - w % patch_size) % patch_size  # Calculate padding for width
    padded_img = np.pad(image, ((0, pad_h), (0, pad_w), (0, 0)), mode='constant')  # Add padding
    image_patches = patchify(padded_img, (patch_size, patch_size, 3), step=patch_size)  # Split image into patches

    predictions = []
    for i in range(image_patches.shape[0]):
        for j in range(image_patches.shape[1]):
            patch = image_patches[i, j, 0] / 255.0  # Normalize patch
            patch = np.expand_dims(patch, axis=0)  # Add batch dimension
            pred = model.predict(patch)  # Predict using the model
            predictions.append(pred[0, :, :, 0])  # Store the prediction

    predictions = np.array(predictions)
    predictions = predictions.reshape(image_patches.shape[0], image_patches.shape[1], patch_size, patch_size)
    root_mask = unpatchify(predictions, padded_img.shape[:2])  # Combine patches into a full mask
    cleaned_mask = mask_morph((root_mask > 0.5).astype(np.uint8))[:h, :w]  # Clean and crop the mask
    
    mask_path = os.path.join(save_path, f"{image_name}_mask.png")
    cv2.imwrite(mask_path, (cleaned_mask * 255).astype(np.uint8))
    print(f"Saved mask: {mask_path}")
    
    return cleaned_mask

# Main pipeline
output_data = []
for image_file in sorted(os.listdir(dataset_dir)):
    if image_file.endswith(".png"):
        img_path = os.path.join(dataset_dir, image_file)
        image = cv2.imread(img_path)
        print(f"Processing {image_file}...")

        processed_image = petri_dish_border(image)
        cv2.imwrite(os.path.join(results_dir, f"{image_file.split('.')[0]}_processed.png"), processed_image)

        mask = predict_roots(processed_image, model, results_dir, image_file.split('.')[0])
        cv2.imwrite(os.path.join(results_dir, f"{image_file.split('.')[0]}_cleaned_mask.png"), (mask * 255).astype(np.uint8))

        sections = mask_split(mask, processed_image.shape[1], plants_per_image)

        for idx, section_mask in enumerate(sections, start=1):
            cv2.imwrite(os.path.join(results_dir, f"{image_file.split('.')[0]}_section_{idx}.png"), (section_mask * 255).astype(np.uint8))
            root_length = int(round(root_skeleton_length(section_mask)))
            output_data.append({
                "Plant ID": f"{image_file.split('.')[0].lower()}_plant_{idx}",
                "Length (px)": root_length
            })
            print(f"Plant {idx}: Root length = {root_length}")

csv_path = os.path.join(results_dir, "kaggle7_pi.csv")
pd.DataFrame(output_data).to_csv(csv_path, index=False)
print(f"Results and masks saved to {results_dir}")

readme_path = os.path.join(results_dir, "README.md")
with open(readme_path, "w") as f:
    f.write("# Root Detection Pipeline\n\n")
    f.write("This project detects and measures plant root lengths from images of Petri dishes.\n\n")
    f.write("## Pipeline Steps\n")
    f.write("1. **Preprocessing**: Removes black edges from the input image to isolate the Petri dish.\n")
    f.write("2. **Root Mask Prediction**: Predicts root masks using a trained U-Net model.\n")
    f.write("3. **Division into Sections**: Splits the mask into equal-width sections for each plant.\n")
    f.write("4. **Root Length Calculation**: Measures the primary root length using skeletonization and graph analysis.\n\n")
    f.write("## Outputs\n")
    f.write("- **Masks**: Saved as PNG files in the results folder.\n")
    f.write("- **CSV File**: Contains root lengths for all plants.\n")
    f.write("\nRun the pipeline and inspect the `results_output` folder for outputs.\n")

print(f"README generated at {readme_path}")


Processing test_image_1.png...
1/1 [==============================] - 0s 75ms/step
Saved mask: kaggle7_pi\test_image_1_mask.png
Plant 1: Root length = 569
Plant 2: Root length = 0
Plant 3: Root length = 262
Plant 4: Root length = 0
Plant 5: Root length = 23
Processing test_image_10.png...
1/1 [==============================] - 0s 79ms/step
Saved mask: kaggle7_pi\test_image_10_mask.png
Plant 1: Root length = 1115
Plant 2: Root length = 1673
Plant 3: Root length = 1256
Plant 4: Root length = 1333
Plant 5: Root length = 1114
Processing test_image_11.png...
1/1 [==============================] - 0s 80ms/step
Saved mask: kaggle7_pi\test_image_11_mask.png
Plant 1: Root length = 1081
Plant 2: Root length = 1415
Plant 3: Root length = 1466
Plant 4: Root length = 1548
Plant 5: Root length = 1235
Processing test_image_12.png...
1/1 [==============================] - 0s 81ms/step
Saved mask: kaggle7_pi\test_image_12_mask.png
Plant 1: Root length = 1011
Plant 2: Root length = 1081
Plant 3: Root le